In [ ]:
# %% [markdown]
# # 电影推荐系统 - 探索性数据分析（EDA）
# 
# 本 Notebook 用于分析 MovieLens 100K 数据集，理解数据分布和特征，为后续建模提供依据。

# %%
# ---------------------- 1. 初始化：导入库 + 设置中文字体 ----------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Jupyter Notebook 内嵌显示图片
%matplotlib inline

# 设置中文字体（避免中文乱码）
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows 系统用 SimHei
# plt.rcParams['font.sans-serif'] = ['Heiti TC']  # Mac 系统用 HeiTi TC
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# %% [markdown]
# ## 2. 数据加载与初步查看

# %%
# ---------------------- 2.1 加载数据 ----------------------
# 注意：因为 notebook 在 notebooks/ 文件夹下，所以数据路径是 ../data/ml-100k/
data_dir = "../data/ml-100k/"

# 加载评分数据（u.data）
ratings = pd.read_csv(
    f"{data_dir}u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

# 加载电影信息数据（u.item）
genre_cols = [
    "Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]
movies = pd.read_csv(
    f"{data_dir}u.item",
    sep="|",
    encoding="latin-1",
    usecols=[0, 1, 2] + list(range(5, 24)),
    names=["movie_id", "title", "release_date", "unknown"] + genre_cols
)
movies = movies.drop("unknown", axis=1)  # 删除无用的 "unknown" 列

# %%
# ---------------------- 2.2 查看数据基本信息 ----------------------
print("=== 评分数据（ratings）基本信息 ===")
print(ratings.info())
print("\n=== 评分数据前 5 行 ===")
print(ratings.head())

print("\n" + "="*50 + "\n")

print("=== 电影信息数据（movies）基本信息 ===")
print(movies.info())
print("\n=== 电影信息数据前 5 行 ===")
print(movies.head())

# %% [markdown]
# ## 3. 数据简单预处理（为 EDA 做准备）

# %%
# ---------------------- 3.1 处理电影上映年份 ----------------------
# 从 release_date 提取年份，填充缺失值
movies["release_year"] = pd.to_datetime(movies["release_date"]).dt.year
movies["release_year"] = movies["release_year"].fillna(movies["release_year"].median())
movies = movies.drop("release_date", axis=1)

# ---------------------- 3.2 对齐数据（确保评分和电影的 movie_id 一致） ----------------------
common_movies = list(set(ratings["movie_id"]) & set(movies["movie_id"]))
ratings = ratings[ratings["movie_id"].isin(common_movies)]
movies = movies[movies["movie_id"].isin(common_movies)]

print(f"清洗后：\n- 评分记录数：{len(ratings)}\n- 用户数：{ratings['user_id'].nunique()}\n- 电影数：{ratings['movie_id'].nunique()}")

# %% [markdown]
# ## 4. 可视化分析

# %% [markdown]
# ### 4.1 用户评分分布
# 分析用户的评分偏好，看看大部分用户给多少分。

# %%
plt.figure(figsize=(8, 5))
sns.countplot(x="rating", data=ratings, palette="viridis")
plt.title("用户评分分布", fontsize=14)
plt.xlabel("评分（1-5分）", fontsize=12)
plt.ylabel("评分数量", fontsize=12)
# 在柱子上标注数量
for p in plt.gca().patches:
    plt.gca().text(p.get_x() + p.get_width()/2, p.get_height() + 100, 
                   f"{int(p.get_height())}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

# %% [markdown]
# **结论**：
# - 4分是最常见的评分，其次是3分和5分；
# - 1分和2分的评分较少，说明用户倾向于给电影中等偏上的评价。

# %% [markdown]
# ### 4.2 电影类型分布
# 分析数据集中哪些类型的电影最多。

# %%
# 统计每个类型的电影数量
genre_count = movies[genre_cols].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=genre_count.index, y=genre_count.values, palette="viridis")
plt.title("电影类型数量分布", fontsize=14)
plt.xlabel("电影类型", fontsize=12)
plt.ylabel("电影数量", fontsize=12)
plt.xticks(rotation=45, ha="right")  # 旋转x轴标签，避免重叠
# 在柱子上标注数量
for i, v in enumerate(genre_count.values):
    plt.text(i, v + 5, str(v), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

# %% [markdown]
# **结论**：
# - Drama（剧情片）和 Comedy（喜剧片）是数量最多的两个类型；
# - 其次是 Action（动作片）和 Thriller（惊悚片）；
# - 纪录片（Documentary）和奇幻片（Fantasy）数量较少。

# %% [markdown]
# ### 4.3 用户评分数量分布
# 分析每个用户的评分习惯，看看大部分用户评了多少部电影。

# %%
# 统计每个用户的评分数量
user_rating_count = ratings["user_id"].value_counts()

plt.figure(figsize=(10, 5))
sns.histplot(user_rating_count, bins=30, kde=True, color="skyblue")
plt.title("用户评分数量分布", fontsize=14)
plt.xlabel("用户评分数量（部）", fontsize=12)
plt.ylabel("用户数", fontsize=12)
plt.axvline(user_rating_count.mean(), color="red", linestyle="--", label=f"平均值：{user_rating_count.mean():.1f}")
plt.axvline(user_rating_count.median(), color="orange", linestyle="--", label=f"中位数：{user_rating_count.median():.1f}")
plt.legend()
plt.tight_layout()
plt.show()

# %% [markdown]
# **结论**：
# - 大部分用户的评分数量在 5-100 部之间；
# - 平均值（约 100 部）高于中位数（约 50 部），说明有少量“超级用户”评了很多电影，拉高了平均值。

# %% [markdown]
# ### 4.4 电影被评分数量分布
# 分析每部电影的受欢迎程度，看看大部分电影被多少人评过分。

# %%
# 统计每部电影的被评分数量
movie_rating_count = ratings["movie_id"].value_counts()

plt.figure(figsize=(10, 5))
sns.histplot(movie_rating_count, bins=30, kde=True, color="lightgreen")
plt.title("电影被评分数量分布", fontsize=14)
plt.xlabel("电影被评分数量（次）", fontsize=12)
plt.ylabel("电影数", fontsize=12)
plt.axvline(movie_rating_count.mean(), color="red", linestyle="--", label=f"平均值：{movie_rating_count.mean():.1f}")
plt.axvline(movie_rating_count.median(), color="orange", linestyle="--", label=f"中位数：{movie_rating_count.median():.1f}")
plt.legend()
plt.tight_layout()
plt.show()

# %% [markdown]
# **结论**：
# - 大部分电影的被评分数量在 10-200 次之间；
# - 同样存在“热门电影”（被评分次数远高于平均值），比如《肖申克的救赎》《星球大战》等经典电影。

# %% [markdown]
# ### 4.5 电影上映年份分布
# 分析数据集中电影的上映时间趋势。

# %%
plt.figure(figsize=(12, 5))
sns.histplot(movies["release_year"], bins=30, kde=True, color="salmon")
plt.title("电影上映年份分布", fontsize=14)
plt.xlabel("上映年份", fontsize=12)
plt.ylabel("电影数量", fontsize=12)
plt.tight_layout()
plt.show()

# %% [markdown]
# **结论**：
# - 数据集中的电影主要集中在 1990-1997 年之间；
# - 早期（1950年之前）的电影数量较少。

# %% [markdown]
# ## 5. EDA 总结
# 
# 通过以上分析，我们对数据集有了以下了解：
# 1. **评分偏好**：用户倾向于给 3-5 分的中等偏上评价，4 分最多；
# 2. **电影类型**：Drama（剧情）和 Comedy（喜剧）是最主流的类型；
# 3. **用户行为**：大部分用户评分 5-100 部，但存在少量“超级用户”；
# 4. **电影热度**：大部分电影被评分 10-200 次，但存在少量“热门经典电影”；
# 5. **时间趋势**：电影主要集中在 90 年代。
# 
# 这些结论将为后续的特征工程和模型选择提供依据！